# CoAD on Kaggle: M11 (part 16 of CoAD_reimplementation.ipynb)

Needs the saved models: the dataset must include `runs/reimpl/`. Run all cells (or Save Version > Save & Run All). Download `to_drive.zip` from the Output tab at the end; it holds eda/m11_split.csv too.

In [ ]:
import json, os, random, shutil, subprocess, sys, time, zipfile
from contextlib import nullcontext
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# Kaggle instead of Google Drive: the Drive folder CoAD_project is attached as a private dataset (read-only).
# Small folders are copied to a writable CoAD_project; the data archive and the saved models are read in place.
# Result files of earlier versions of this notebook, attached as inputs, are added back so finished runs are skipped.
KAGGLE_TAG = 'kaggle-m11'
INPUT = Path('/kaggle/input')
candidates = [p.parents[1] for p in INPUT.rglob('env/commit.txt')
              if (p.parents[1] / 'data_source' / 'TSB-AD-U.zip').is_file()]
assert candidates, 'Attach the dataset with the CoAD_project folder (env/, eda/, results/, data_source/)'
source = max(candidates, key=lambda d: (d / 'runs' / 'reimpl').is_dir())
STORE = Path('/kaggle/working/CoAD_project')
if not STORE.exists():
    for sub in ['env', 'eda', 'results']:
        shutil.copytree(source / sub, STORE / sub)
    (STORE / 'data_source').symlink_to(source / 'data_source', target_is_directory=True)
    (STORE / 'runs').mkdir()
    if (source / 'runs' / 'reimpl').is_dir():
        (STORE / 'runs' / 'reimpl').symlink_to(source / 'runs' / 'reimpl', target_is_directory=True)
    latest = {}
    for csv in INPUT.rglob('results/*.csv'):
        if csv.name not in latest or csv.stat().st_size > latest[csv.name].stat().st_size:
            latest[csv.name] = csv
    for name, csv in latest.items():
        shutil.copy2(csv, STORE / 'results' / name)
print('Dataset:', source, '| result files:', len(list((STORE / 'results').glob('*.csv'))),
      '| saved models:', (STORE / 'runs' / 'reimpl').exists())
REPO = Path('/tmp/CoAD')
commit = (STORE / 'env' / 'commit.txt').read_text(encoding='utf-8').strip()
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/richard-tang199/CoAD.git', str(REPO)], check=True)
subprocess.run(['git', 'checkout', '-q', commit], cwd=REPO, check=True)

CHECK_IMPORTS = 'import arch, stumpy, tslearn, sktime, torch_geometric, statsmodels, numba'

def deps_ok():
    return subprocess.run([sys.executable, '-c', CHECK_IMPORTS], capture_output=True).returncode == 0

if not deps_ok():
    install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')],
                             capture_output=True, text=True)
    (STORE / 'env' / 'install_reimpl.log').write_text(install.stdout + install.stderr, encoding='utf-8')
    assert install.returncode == 0, 'Installation failed; inspect env/install_reimpl.log'
assert deps_ok(), 'Some packages still cannot be imported; inspect env/install_reimpl.log'
print('Commit', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO, text=True).strip(),
      '| packages OK | GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
archive = STORE / 'data_source' / 'TSB-AD-U.zip'
RAW = REPO / 'dataset' / 'TSB-AD' / 'raw'
RAW.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for m in z.namelist():
        if m.startswith('TSB-AD-U/') and m.endswith('.csv') and '..' not in m:
            target = RAW / Path(m).name
            if not target.exists():
                with z.open(m) as src, target.open('wb') as dst:
                    shutil.copyfileobj(src, dst)
assert len(list(RAW.iterdir())) == 870, 'raw/ must contain exactly 870 CSVs'

os.chdir(REPO)                       # load_dataset reads the relative path dataset/TSB-AD/raw
sys.path.insert(0, str(REPO))
from toolkit.utils import load_dataset
from evaluation.evaluator import evaluate
from model.frequency import time_to_timefreq       # reference for test T2 only
from toolkit import distort as reference_distort   # reference for test T1 only
from model.patch_soft_mask import PatchFrequencyMask  # reference for test T4 only

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPORTS = STORE / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
print('Data: 870 files | device:', DEVICE)

In [ ]:
FAITHFUL_TO_CODE = False   # False: follow the paper (main version); True: follow the authors' code (D1-D21)
SEEDS = [46, 47, 48]
PATCH = 8                  # P
WINDOW_MULTIPLE = 4        # T = 4 x dominant period
N_FFT = 4                  # K, STFT size
LAMBDA = 10                # weight of the reconstruction loss
LR = 0.002
EPOCHS = 300
BATCH_SIZE = 128

def use_code_details(faithful=None):
    """Per-call override of FAITHFUL_TO_CODE (None = use the global switch)."""
    return FAITHFUL_TO_CODE if faithful is None else faithful

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def load_series(group):
    """One TSB-AD series (min-max scaled on train) plus the window settings of the authors' code."""
    train, test, label, period = [values[0] for values in load_dataset('TSB-AD', [str(group)])]
    multiple = WINDOW_MULTIPLE if len(train) // period > WINDOW_MULTIPLE else len(train) // period
    window = (period * multiple) // PATCH * PATCH
    period = min(period, window)
    n = len(train)
    train_stride = 2 if n < 1000 else 6 if n < 10000 else 10 if n < 60000 else PATCH if n < 100000 else period
    return dict(group=group, train=train, test=test, label=label,
                period=period, window=window, train_stride=train_stride)

print('FAITHFUL_TO_CODE =', FAITHFUL_TO_CODE, '| seeds', SEEDS)

In [ ]:
manifest64 = pd.read_csv(STORE / 'eda' / 'manifest_tsbad64.csv')
reimpl_manifest_file = STORE / 'eda' / 'manifest_reimpl.csv'
if not reimpl_manifest_file.exists():
    rng = np.random.default_rng(2026)
    chosen = [236]
    for family in ['MGAB', 'SVDB', 'IOPS', 'TODS']:
        chosen.append(int(rng.choice(sorted(manifest64.loc[manifest64['family'] == family, 'group']))))
    rows = manifest64.set_index('group').loc[chosen].reset_index()
    rows['selection_reason'] = ['Pilot series of the reproduction'] + ['default_rng(2026), one per family'] * 4
    rows.to_csv(reimpl_manifest_file, index=False)
REIMPL_MANIFEST = pd.read_csv(reimpl_manifest_file)
REIMPL_GROUPS = REIMPL_MANIFEST['group'].tolist()

overview = []
for g, family in zip(REIMPL_GROUPS, REIMPL_MANIFEST['family']):
    s = load_series(g)
    overview.append({'group': g, 'family': family, 'n_train': len(s['train']), 'n_test': len(s['test']),
                     'anomaly_%': round(100 * s['label'].mean(), 2), 'period': s['period'],
                     'window': s['window'], 'train_stride': s['train_stride']})
display(pd.DataFrame(overview))

In [ ]:
DISTORTIONS = ['uniform', 'scale', 'jitter', 'mirror']

def distort(x, period, kind, faithful=None):
    """x: clean windows (B, T, 1). Returns (distorted copy, point labels (B, T))."""
    faithful = use_code_details(faithful)
    x = x.clone()
    B, T, _ = x.shape
    n = int(period * np.random.uniform(0.1, 1))           # segment length, one draw per batch
    if n == 0:
        n = 2 if kind == 'uniform' else 1
    if n >= T:
        n = T - 2
    start = torch.randint(0, T - n, (B, 1))               # segment start, one per window
    if faithful and kind == 'mirror':
        torch.randint(0, T - n, (B, 1))                   # unused draw of the authors' mirror_flip: keeps the RNG stream aligned
    rows = torch.arange(B)[:, None]

    if kind == 'scale':
        factor = np.random.randint(2, 5)
        if faithful or np.random.rand() < 0.5:            # lengthen: first n / factor points, each repeated
            n = n // factor * factor
            source = start + torch.arange(n) // factor
        else:                                             # shorten: every factor-th point of a longer span
            source = (start + torch.arange(n) * factor).clamp(max=T - 1)
    cols = start + torch.arange(n)
    segment = x[rows, cols]                               # (B, n, 1)

    if kind == 'uniform':
        if faithful:
            x[rows, cols] = x[rows, start]
        else:
            low, high = segment.min(1, keepdim=True).values, segment.max(1, keepdim=True).values
            x[rows, cols] = low + (high - low) * torch.rand(B, 1, 1, device=x.device)
    elif kind == 'scale':
        x[rows, cols] = x[rows, source]
    elif kind == 'jitter':
        if faithful:
            std = 0.05 * (x.max(1, keepdim=True).values - x.min(1, keepdim=True).values)
        else:
            std = 0.1 ** 0.5
        x[rows, cols] = segment + torch.randn(B, n, 1, device=x.device) * std
    elif kind == 'mirror':
        if faithful or np.random.rand() < 0.5:            # across the y-axis: reverse time
            x[rows, cols] = torch.flip(segment, [1])
        else:                                             # across the x-axis: mirror values inside the segment range
            x[rows, cols] = segment.min(1, keepdim=True).values + segment.max(1, keepdim=True).values - segment
    else:
        raise ValueError(f'unknown distortion {kind}')

    labels = torch.zeros(B, T, device=x.device)
    labels[rows, cols] = 1
    return x, labels

In [ ]:
def stft_features(x):
    """x: (B, T) -> (B, T, 2 * bins): real and imaginary part of every bin, one frame per time point."""
    spectrum = torch.stft(x, N_FFT, hop_length=N_FFT // 4, window=torch.hann_window(N_FFT, device=x.device),
                          normalized=True, onesided=True, return_complex=True)
    spectrum = torch.view_as_real(spectrum)            # (B, bins, frames, 2)
    spectrum = spectrum.permute(0, 2, 1, 3).flatten(2)  # (B, frames, bins * 2)
    return spectrum[:, :x.shape[1]]                     # frames = T + 1 (centred STFT); keep T

def make_patches(x, x_freq):
    """x: (B, T), x_freq: (B, T, F) -> time patches (B, N, P) and frequency patches (B, N, P * F)."""
    B, T = x.shape
    return x.reshape(B, T // PATCH, PATCH), x_freq.reshape(B, T // PATCH, -1)

In [ ]:
def head(in_size, hidden, faithful):
    """Patch classifier: linear in the paper (Eq. 2, 3, 8), two-layer MLP in the code (D4)."""
    if faithful:
        return nn.Sequential(nn.Linear(in_size, hidden), nn.SELU(), nn.Linear(hidden, 1, bias=False))
    return nn.Sequential(nn.Linear(in_size, 1))

def gru(size, faithful):
    """Bi-GRU encoder (Section 3.2): 2 layers and dropout 0.1 in the code, 3 layers in the paper (D1, D2)."""
    return nn.GRU(size, size, num_layers=2 if faithful else 3, batch_first=True, bidirectional=True,
                  dropout=0.1 if faithful else 0.0)

def init_linear(module):
    """Xavier-uniform weights and zero biases for every linear layer, as in the authors' code."""
    for layer in module.modules():
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)
            if layer.bias is not None:
                nn.init.zeros_(layer.bias)


class TimeFreqClassifier(nn.Module):
    """Module 3 (Eq. 1-4) and the residual classifier of module 5 (Eq. 7-8)."""

    def __init__(self, faithful):
        super().__init__()
        self.faithful = faithful
        freq_size = PATCH * 2 * (N_FFT // 2 + 1)                  # 8 points x 3 bins x (real, imaginary) = 48
        hidden = 3 * freq_size if faithful else 24                # D5
        activation = [nn.SELU()] if faithful else []              # D18
        self.proj_freq = nn.Sequential(nn.Linear(freq_size, hidden), *activation)   # W_f^p
        self.proj_time = nn.Sequential(nn.Linear(PATCH, hidden), *activation)       # W_t^p
        self.encoder = gru(hidden, faithful)                      # shared by both branches (D3)
        self.classifier_time = head(2 * hidden, hidden, faithful)       # W_t (created first, as in the authors' code,
        self.classifier_freq = head(2 * hidden, hidden, faithful)       # W_f  so a seed gives the same initial weights)
        self.classifier_residual = head(2 * hidden, hidden, faithful)   # W_r, shared by both residual branches
        init_linear(self)

    def features(self, x):
        """x: (B, T) -> encoder outputs of the frequency and of the time patches, each (B, N, 2H)."""
        time_patches, freq_patches = make_patches(x, stft_features(x))
        projected = torch.cat([self.proj_freq(freq_patches), self.proj_time(time_patches)])
        encoded, _ = self.encoder(projected)
        return encoded.chunk(2)

    def forward(self, x):
        """Eq. 1-4: logit of A = max(A_f, A_t), (B, N, 1). Max of logits = max of probabilities (sigmoid is increasing)."""
        h_freq, h_time = self.features(x)
        return torch.maximum(self.classifier_freq(h_freq), self.classifier_time(h_time))

    def residual_logit(self, x, x_rec):
        """Eq. 7-8: logit of A^r from the feature difference between the input and its reconstruction."""
        h_freq, h_time = self.features(torch.cat([x, x_rec]))
        (freq_in, freq_rec), (time_in, time_rec) = h_freq.chunk(2), h_time.chunk(2)
        return torch.maximum(self.classifier_residual(freq_in - freq_rec),
                             self.classifier_residual(time_in - time_rec))


class SoftMaskMAE(nn.Module):
    """Module 4 (Eq. 5-6): patch embeddings softly replaced by a learned mask embedding, then reconstructed."""

    def __init__(self, faithful):
        super().__init__()
        hidden = 3 * PATCH                                        # 24 in both modes
        self.proj = nn.Linear(PATCH, hidden)                      # W_m
        self.act = nn.SELU() if faithful else nn.Identity()       # D18
        self.encoder = gru(hidden, faithful)
        self.head = nn.Linear(2 * hidden, PATCH)                  # W_o
        self.mask_embedding = nn.Parameter(torch.randn(hidden))   # E_mask, one vector repeated over patches (D19)
        init_linear(self)

    def forward(self, x, prob):
        """x: (B, T), prob: patch anomaly probability (B, N, 1) -> reconstruction X_r (B, T)."""
        B, T = x.shape
        embedding = self.act(self.proj(x.reshape(B, T // PATCH, PATCH)))
        mixed = prob * self.mask_embedding + (1 - prob) * embedding    # Eq. 5
        encoded, _ = self.encoder(mixed)
        return self.head(encoded).reshape(B, T)                         # Eq. 6


class CoADReimpl(nn.Module):
    def __init__(self, faithful=None):
        super().__init__()
        self.faithful = use_code_details(faithful)
        self.classifier = TimeFreqClassifier(self.faithful)
        self.mae = SoftMaskMAE(self.faithful)

    def forward(self, x):
        """Training pass. x: (B, T) -> logits of A and A^r (B, N, 1), reconstruction X_r (B, T)."""
        logit = self.classifier(x)
        x_rec = self.mae(x, torch.sigmoid(logit))
        return logit, self.classifier.residual_logit(x, x_rec), x_rec

    @torch.no_grad()
    def predict(self, x):
        """Inference (call .eval() first). x: (B, T) -> A_c (B, N, 1) (Eq. 9) and the reconstruction (B, T)."""
        prob = torch.sigmoid(self.classifier(x))
        x_rec = self.mae(x, prob)
        prob = (prob + torch.sigmoid(self.classifier.residual_logit(x, x_rec))) / 2
        if self.faithful:                   # D10: reconstruct again, masked with A_c
            x_rec = self.mae(x, prob)
        return prob, x_rec

    def load_official(self, official):
        """Copy the weights of an authors' PatchFrequencyMask (code mode; test T4)."""
        renamed = {k.replace('detection_net.', 'classifier.').replace('reconstruction_net.', 'mae.'): v
                   for k, v in official.state_dict().items()
                   if k.startswith(('detection_net.', 'reconstruction_net.'))}
        self.load_state_dict(renamed)

In [ ]:
def make_windows(values, window, stride):
    """values (L,) -> windows (M, window) float32 and the front padding length."""
    L = len(values)
    pad = 0 if stride == 1 or (L - window) % stride == 0 else stride - (L - window) % stride
    padded = np.concatenate([np.zeros(pad), values])
    starts = np.arange(0, len(padded) - window + 1, stride)
    return np.stack([padded[s:s + window] for s in starts]).astype(np.float32), pad

def windows_to_series(window_values, length, stride, pad):
    """window_values (M, window) -> (length,): mean over the overlapping windows, padding removed."""
    total, count = np.zeros(pad + length), np.zeros(pad + length)
    for i, values in enumerate(window_values):
        total[i * stride:i * stride + len(values)] += values
        count[i * stride:i * stride + len(values)] += 1
    return total[pad:] / count[pad:]

def patch_labels(labels):
    """Point labels (B, T) -> patch labels (B, N, 1)."""
    B, T = labels.shape
    return labels.reshape(B, T // PATCH, PATCH).amax(-1, keepdim=True)

def training_batch(clean, period, faithful, code_distortion=None):
    """Clean windows (B, T) -> model input, point labels, reconstruction target.
    code_distortion (M6-C only): distortion details D12-D15 chosen apart from the batch layout D9."""
    code_distortion = faithful if code_distortion is None else code_distortion
    distorted, labels = distort(clean.unsqueeze(-1), period, random.choice(DISTORTIONS), code_distortion)
    distorted = distorted.squeeze(-1)
    if faithful:                                    # D9: clean copy + distorted copy
        return torch.cat([clean, distorted]), torch.cat([torch.zeros_like(labels), labels]), torch.cat([clean, clean])
    return distorted, labels, clean

def coad_loss(model, x, labels, target, code_loss=None):
    """Eq. 10. Returns (total, classification term, reconstruction term).
    code_loss (M6-C only): loss details D6-D8, D20 chosen apart from the model."""
    logit, residual_logit, x_rec = model(x)
    y = patch_labels(labels)
    if (model.faithful if code_loss is None else code_loss):
        pos_weight = torch.tensor([2.0], device=x.device)                                    # D6
        classify = (F.binary_cross_entropy_with_logits(logit, y, pos_weight=pos_weight) +
                    F.binary_cross_entropy_with_logits(residual_logit, y, pos_weight=pos_weight)) / 2   # D7
        error = (x_rec - target).pow(2).reshape(y.shape[0], y.shape[1], PATCH)
        anomalous = (error * y).sum() / (y.sum() * PATCH + 1e-7)
        normal = (error * (1 - y)).sum() / ((1 - y).sum() * PATCH + 1e-7)
        recon = 0.5 * anomalous + 0.5 * normal                                               # D8
    else:
        prob = (torch.sigmoid(logit) + torch.sigmoid(residual_logit)) / 2                    # A_c, Eq. 9
        classify = F.binary_cross_entropy(prob, y)
        recon = PATCH * F.mse_loss(x_rec, target)      # Eq. 10: squared norm summed over a patch, averaged over patches (D20)
    return classify + LAMBDA * recon, classify, recon

In [ ]:
def train_model(series, faithful=None, seed=46, epochs=None, log_every=50, groups=None):
    """Train CoADReimpl on the training part of one series. Returns the model and the mean losses per epoch.
    groups (M6-C only): set of code-only detail groups (section 12) applied on top of paper mode."""
    epochs = EPOCHS if epochs is None else epochs
    set_seed(seed)
    if groups is None:
        faithful = use_code_details(faithful)
        code_distortion = faithful
        model = CoADReimpl(faithful).to(DEVICE)
    else:
        faithful, code_distortion = 'training' in groups, 'distortion' in groups
        model = HybridCoAD(groups).to(DEVICE)
    windows, _ = make_windows(series['train'], series['window'], series['train_stride'])
    loader = DataLoader(TensorDataset(torch.from_numpy(windows)), batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.99) if faithful else None   # D16
    use_amp = faithful and DEVICE.type == 'cuda'                                                            # D16
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    history = []
    for epoch in range(epochs):
        model.train()
        sums = np.zeros(3)
        for (clean,) in loader:
            x, labels, target = training_batch(clean.to(DEVICE), series['period'], faithful, code_distortion)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16) if use_amp else nullcontext():
                losses = coad_loss(model, x, labels, target, faithful)
            scaler.scale(losses[0]).backward()
            scaler.step(optimizer)
            scaler.update()
            sums += [value.item() for value in losses]
        if scheduler:
            scheduler.step()
        history.append(dict(zip(['loss', 'classify', 'recon'], sums / len(loader))))
        if log_every and (epoch + 1) % log_every == 0:
            print(f'  epoch {epoch + 1}/{epochs}: ' + ', '.join(f'{k} {v:.4f}' for k, v in history[-1].items()),
                  flush=True)
    return model, pd.DataFrame(history)

In [ ]:
@torch.no_grad()
def predict_series(model, values, window, stride):
    """Whole series -> per-point A_c and reconstruction, averaged over overlapping windows."""
    windows, pad = make_windows(values, window, stride)
    model.eval()
    probs, recs = [], []
    for batch in torch.from_numpy(windows).split(2 * BATCH_SIZE):
        prob, rec = model.predict(batch.to(DEVICE))
        probs.append(prob.repeat_interleave(PATCH, dim=1).squeeze(-1).cpu())
        recs.append(rec.cpu())
    return (windows_to_series(torch.cat(probs).numpy(), len(values), stride, pad),
            windows_to_series(torch.cat(recs).numpy(), len(values), stride, pad))

def moving_average(score, width):
    """Centred moving average keeping the length (np.convolve 'same', as in the authors' code)."""
    return np.convolve(score, np.ones(width) / width, mode='same')

def anomaly_score(model, series):
    """Per-point anomaly score of the test series."""
    period, window = series['period'], series['window']
    test_prob, test_rec = predict_series(model, series['test'], window, period)
    if not model.faithful:
        return moving_average(test_prob + np.abs(series['test'] - test_rec), period)    # Eq. 11
    train_prob, train_rec = predict_series(model, series['train'], window, period)      # D11
    recon = np.abs(np.abs(series['test'] - test_rec) - np.abs(series['train'] - train_rec).mean())
    classify = np.abs(test_prob - train_prob.mean())
    recon, classify = moving_average(recon, period), moving_average(classify, period)
    recon[:period] = classify[:period] = 0
    return (recon + classify) / 2

METRIC_KEYS = ['f1_score', 'auc_prc', 'auc_roc', 'R_AUC_ROC', 'R_AUC_PR', 'VUS_ROC', 'VUS_PR']

def score_metrics(score, series):
    """Checks of test T7, then the repository evaluator (values in 0-1)."""
    assert len(score) == len(series['label']), 'score and label lengths differ'
    assert np.isfinite(score).all(), 'score contains NaN or Inf'
    result = evaluate(ground_truth=series['label'], anomaly_scores=score, subsequence=series['period'], mode='test')
    row = {k: float(result.point_wise[k]) for k in METRIC_KEYS[:3]}
    row.update({k: float(result.vus[k]) for k in METRIC_KEYS[3:]})
    return row

def plot_scores(series, scores, path=None):
    """Test series on top, one min-max scaled score per row below; labelled anomalies shaded."""
    fig, axes = plt.subplots(len(scores) + 1, 1, figsize=(14, 1.6 * (len(scores) + 1)), sharex=True, squeeze=False)
    axes = axes[:, 0]
    rows = [('test data', series['test'])] + list(scores.items())
    for ax, (name, values) in zip(axes, rows):
        values = np.asarray(values, dtype=float)
        ax.plot((values - values.min()) / (values.max() - values.min() + 1e-12), lw=0.6)
        ax.fill_between(np.arange(len(values)), 0, 1, where=series['label'] == 1, color='C3', alpha=0.25,
                        transform=ax.get_xaxis_transform())
        ax.set_ylabel(name, rotation=0, ha='right', fontsize=8)
        ax.set_yticks([])
    fig.suptitle(f"group {series['group']}")
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=120)
    plt.show()

In [ ]:
RESULTS_DIR = STORE / 'results'
RUNS = STORE / 'runs' / 'reimpl'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)

SESSION_TAG = KAGGLE_TAG            # unique per notebook copy running at the same time
RESULTS = RESULTS_DIR / f'reimpl_results_{SESSION_TAG}.csv'
RUN_MODES = ['paper', 'code']
RUN_GROUPS = REIMPL_GROUPS + [g for g in manifest64['group'] if g not in REIMPL_GROUPS]   # 5 chosen series first, then the other 59
RUN_SEEDS = SEEDS
TIME_BUDGET_MIN = 600   # Kaggle: stop well before the 12 h limit            # no new run after this many minutes; re-run 7.2 to continue
COLUMNS = ['run_id', 'mode', 'group', 'seed', 'status', 'seconds', 'error'] + METRIC_KEYS

def read_reimpl_results():
    """Rows of every session, oldest run first."""
    files = sorted(RESULTS_DIR.glob('reimpl_results*.csv'))
    if not files:
        return pd.DataFrame(columns=COLUMNS)
    rows = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    return rows.sort_values('run_id', kind='stable').reset_index(drop=True)

def run_reimpl(mode, group, seed):
    run_id = f"{datetime.now(timezone.utc):%Y%m%dT%H%M%S}_{mode}_{group}_s{seed}"
    out = RUNS / run_id
    out.mkdir(parents=True)
    row = dict(run_id=run_id, mode=mode, group=group, seed=seed)
    start = time.time()
    try:
        series = load_series(group)
        model, history = train_model(series, faithful=(mode == 'code'), seed=seed, log_every=0)
        score = anomaly_score(model, series)
        metrics = score_metrics(score, series)
        torch.save(model.state_dict(), out / 'model.pt')
        np.save(out / 'scores.npy', score.astype(np.float32))
        history.to_csv(out / 'loss.csv', index=False)
        (out / 'metrics.json').write_text(json.dumps(metrics, indent=2))
        row.update(status='success', **metrics)
    except Exception as error:
        row.update(status='failed', error=f'{type(error).__name__}: {error}'[:300])
    row['seconds'] = round(time.time() - start, 1)
    config = dict(row, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, patch=PATCH, n_fft=N_FFT, lam=LAMBDA,
                  commit=commit, device=str(DEVICE))
    (out / 'config.json').write_text(json.dumps(config, indent=2, default=str))
    pd.DataFrame([row]).reindex(columns=COLUMNS).to_csv(RESULTS, mode='a', header=not RESULTS.exists(), index=False)
    return row

print('Session', SESSION_TAG, '| writes to', RESULTS.name, '|', len(RUN_MODES) * len(RUN_GROUPS) * len(RUN_SEEDS), 'runs planned')

In [ ]:
from scipy.stats import friedmanchisquare, wilcoxon

METRICS4 = {'f1_score': 'F1', 'auc_prc': 'AUC-PR', 'R_AUC_PR': 'R-AUC-PR', 'VUS_PR': 'VUS-PR'}
BASELINES = ['Kmeans', 'KshapeAD', 'SAND', 'Series2Graph', 'MP', 'PCA', 'damp']
groups64 = manifest64['group'].tolist()
family_of = manifest64.set_index('group')['family']

reproduced = pd.concat([pd.read_csv(f) for f in sorted(RESULTS_DIR.glob('tsbad64_results*.csv'))], ignore_index=True)
reproduced = (reproduced.query("status == 'success'").sort_values('run_id')
              .drop_duplicates(['method', 'group'], keep='last'))
reimpl = (read_reimpl_results().query("status == 'success'")
          .drop_duplicates(['mode', 'group', 'seed'], keep='last'))

def metric_matrix(key):
    """64 series x methods, values x100."""
    table = reproduced.pivot(index='group', columns='method', values=key)
    for mode in ['code', 'paper']:
        table[f'CoAD reimpl {mode}'] = reimpl[reimpl['mode'] == mode].groupby('group')[key].mean()
    table = 100 * table.reindex(groups64)[['CoAD', 'CoAD reimpl code', 'CoAD reimpl paper'] + BASELINES]
    assert table.notna().all().all(), f'missing values for {key}: complete the runs first'
    return table

MATRIX = {name: metric_matrix(key) for key, name in METRICS4.items()}
print('64 series x', MATRIX['VUS-PR'].shape[1], 'methods | reimplementation seeds per mode:',
      reimpl.groupby('mode')['seed'].nunique().to_dict())
display(pd.DataFrame({name: m.mean() for name, m in MATRIX.items()}).round(2))

In [ ]:
def holm(pvalues):
    """Holm step-down adjustment."""
    pvalues = np.asarray(pvalues)
    adjusted, running = np.empty(len(pvalues)), 0.0
    for i, idx in enumerate(np.argsort(pvalues)):
        running = max(running, (len(pvalues) - i) * pvalues[idx])
        adjusted[idx] = min(1.0, running)
    return adjusted

def bootstrap_ci(diff, n=10_000, seed=2026):
    """95% interval of the mean difference, resampling series with replacement."""
    rng = np.random.default_rng(seed)
    means = rng.choice(diff, size=(n, len(diff)), replace=True).mean(axis=1)
    return np.percentile(means, [2.5, 97.5])

pairwise = []
for coad in ['CoAD', 'CoAD reimpl code']:
    for name, matrix in MATRIX.items():
        rows = []
        for baseline in BASELINES:
            diff = (matrix[coad] - matrix[baseline]).to_numpy()
            low, high = bootstrap_ci(diff)
            rows.append({'CoAD': coad, 'metric': name, 'baseline': baseline,
                         'CoAD better': int((diff > 0).sum()), 'CoAD worse': int((diff < 0).sum()),
                         'mean diff': diff.mean(), 'CI low': low, 'CI high': high,
                         'p': wilcoxon(diff).pvalue})
        for row, p_holm in zip(rows, holm([r['p'] for r in rows])):
            row['p Holm'] = p_holm
            row['significant'] = p_holm < 0.05
        pairwise += rows
pairwise = pd.DataFrame(pairwise)
pairwise.to_csv(REPORTS / 'm5_pairwise_tests.csv', index=False)
display(pairwise[pairwise['metric'] == 'VUS-PR'].round(3))
print('Not significant after Holm (all metrics):')
display(pairwise[~pairwise['significant']][['CoAD', 'metric', 'baseline', 'CoAD better', 'CoAD worse',
                                            'mean diff', 'p Holm']].round(3))

In [ ]:
@torch.no_grad()
def predict_components(model, values, window, stride):
    """Per-point A (OE), A^r (residual), A_c and the final reconstruction, averaged over overlapping windows.
    Same computation as CoADReimpl.predict (section 4.1), keeping the intermediate tensors."""
    windows, pad = make_windows(values, window, stride)
    model.eval()
    parts = {k: [] for k in ['oe', 'residual', 'combined', 'reconstruction']}
    for batch in torch.from_numpy(windows).split(2 * BATCH_SIZE):
        x = batch.to(DEVICE)
        oe = torch.sigmoid(model.classifier(x))
        first = model.mae(x, oe)
        residual = torch.sigmoid(model.classifier.residual_logit(x, first))
        combined = (oe + residual) / 2
        final = model.mae(x, combined) if model.faithful else first      # D10
        for key, value in [('oe', oe), ('residual', residual), ('combined', combined)]:
            parts[key].append(value.repeat_interleave(PATCH, dim=1).squeeze(-1).cpu())
        parts['reconstruction'].append(final.cpu())
    return {k: windows_to_series(torch.cat(v).numpy(), len(values), stride, pad) for k, v in parts.items()}

@torch.no_grad()
def backup_error(model, values, window, stride, n_groups=4):
    """Module A: |x - reconstruction| when the patch itself is fully masked, independently of OE."""
    windows, pad = make_windows(values, window, stride)
    model.eval()
    out = []
    for batch in torch.from_numpy(windows).split(2 * BATCH_SIZE):
        x = batch.to(DEVICE)
        B, T = x.shape
        index = torch.arange(T // PATCH, device=x.device)
        error = torch.zeros_like(x)
        for j in range(n_groups):
            masked = (index % n_groups == j).float()
            reconstruction = model.mae(x, masked.view(1, -1, 1).expand(B, -1, 1))
            error += (x - reconstruction).abs() * masked.repeat_interleave(PATCH).view(1, T)
        out.append(error.cpu())
    return windows_to_series(torch.cat(out).numpy(), len(values), stride, pad)

def processed(raw_test, raw_train, period, smooth=True, zero_start=True):
    """Authors' code-mode processing of one source: |x - train mean|, moving average, first period zeroed.
    Returns the processed test score and its train counterpart (for calibration)."""
    mean = raw_train.mean()
    out = []
    for raw in (raw_test, raw_train):
        value = np.abs(raw - mean)
        value = moving_average(value, period) if smooth else value.copy()
        if zero_start:
            value[:period] = 0
        out.append(value)
    return out

def model_signals(model, series, values=None, train_parts=None):
    """All point signals of one model on the test series (or on `values`, e.g. a slice with an injected anomaly).
    train_parts: precomputed training-part components, reused by the stress test."""
    period, window = series['period'], series['window']
    values = series['test'] if values is None else values
    test = predict_components(model, values, window, period)
    test['backup'] = backup_error(model, values, window, period)
    if train_parts is None:
        train_parts = predict_components(model, series['train'], window, period)
        train_parts['backup'] = backup_error(model, series['train'], window, period)
    zero = values is series['test']
    error_test, error_train = np.abs(values - test['reconstruction']), np.abs(series['train'] - train_parts['reconstruction'])
    r, r_train = processed(error_test, error_train, period, zero_start=zero)
    c, c_train = processed(test['combined'], train_parts['combined'], period, zero_start=zero)
    b, b_train = processed(test['backup'], train_parts['backup'], period, zero_start=zero)
    r_raw, _ = processed(error_test, error_train, period, smooth=False, zero_start=zero)
    c_raw, _ = processed(test['combined'], train_parts['combined'], period, smooth=False, zero_start=zero)
    s = {name: np.std(t) + 1e-12 for name, t in [('r', r_train), ('c', c_train), ('b', b_train)]}
    signals = {
        'full': (r + c) / 2, 'classify': c, 'reconstruct': r, 'backup': b, 'full_unsmoothed': (r_raw + c_raw) / 2,
        'oe': test['oe'], 'residual': test['residual'], 'error': error_test,
        'calibrated': (r / s['r'] + c / s['c']) / 2,
        'calibrated + backup': (r / s['r'] + c / s['c'] + b / s['b']) / 3,
    }
    return signals, train_parts

EVENT_SIGNALS = ['full', 'classify', 'reconstruct', 'backup', 'full_unsmoothed', 'oe', 'residual', 'error']
SPLIT_SIGNALS = ['full', 'classify', 'reconstruct', 'backup']
VARIANTS = ['calibrated', 'backup', 'calibrated + backup']
# Fixed column orders: rows are appended model by model, so every model must write the same columns in the same order
EVENT_COLUMNS = (['run_id', 'group', 'seed', 'family', 'event', 'start', 'length', 'length_over_period', 'window_share',
                  'amplitude_z'] + [f'{s}_{k}' for s in EVENT_SIGNALS
                                    for k in (['peak', 'mean', 'middle', 'edges'] if s in SPLIT_SIGNALS else ['peak', 'mean'])])
STRESS_COLUMNS = (['group', 'seed', 'family', 'type', 'length_periods', 'offset', 'position']
                  + [f'{s}_{k}' for s in SPLIT_SIGNALS + ['oe'] for k in ['peak', 'mean', 'middle', 'edges']] + ['spill'])

def segments(label):
    edges = np.diff(np.concatenate([[0], np.asarray(label).astype(int), [0]]))
    return list(zip(np.where(edges == 1)[0], np.where(edges == -1)[0]))

def percentile_against(reference):
    """Fraction of the reference (normal) values strictly below each value."""
    ordered = np.sort(reference)
    return lambda v: np.searchsorted(ordered, v, side='left') / len(ordered)

def event_stats(signal, start, end, pct):
    """Peak and mean percentile over an event; middle third vs both edge thirds for events of 3 patches or more."""
    p = pct(signal[start:end])
    row = {'peak': float(pct(np.array([signal[start:end].max()]))[0]), 'mean': float(p.mean())}
    if end - start >= 3 * PATCH:
        third = (end - start) // 3
        row.update(middle=float(p[third:2 * third].mean()), edges=float(np.r_[p[:third], p[2 * third:]].mean()))
    return row

def event_rows(series, signals, base):
    """One row per labelled event. Normal reference: label 0, after the first period."""
    label, test, period = np.asarray(series['label']), series['test'], series['period']
    normal = (label == 0) & (np.arange(len(label)) >= period)
    mu, sd = test[normal].mean(), test[normal].std() + 1e-12
    pcts = {name: percentile_against(signals[name][normal]) for name in EVENT_SIGNALS}
    rows = []
    for k, (a, b) in enumerate(segments(label)):
        row = dict(base, event=k, start=int(a), length=int(b - a), length_over_period=(b - a) / period,
                   window_share=min(1.0, (b - a) / series['window']),
                   amplitude_z=float(np.abs(test[a:b] - mu).max() / sd))
        for name in EVENT_SIGNALS:
            stats = event_stats(signals[name], a, b, pcts[name])
            if name not in SPLIT_SIGNALS:
                stats = {k2: v for k2, v in stats.items() if k2 in ('peak', 'mean')}
            row.update({f'{name}_{k2}': v for k2, v in stats.items()})
        rows.append(row)
    return rows

def alarm_stats(series, score):
    """Top-k points by score, k = number of anomalous points: share that are normal, and of those the share
    within one period of an event (spill-over) rather than far from any event."""
    label, period = np.asarray(series['label']), series['period']
    valid = np.arange(len(label)) >= period
    k = int(label[valid].sum())
    top = np.where(valid)[0][np.argsort(-score[valid])[:k]]
    normal_top = top[label[top] == 0]
    near = np.zeros(len(label), bool)
    for a, b in segments(label):
        near[max(0, a - period):b + period] = True
    return {'top_k_normal_share': len(normal_top) / max(k, 1),
            'normal_alarms_near_share': float(near[normal_top].mean()) if len(normal_top) else np.nan}

In [ ]:
from sklearn.metrics import average_precision_score

M11_SPLIT = STORE / 'eda' / 'm11_split.csv'
if not M11_SPLIT.exists():
    rng = np.random.default_rng(2026)
    rows = []
    for family, d in manifest64.groupby('family', sort=True):
        groups = sorted(int(g) for g in d['group'])
        development = set(int(g) for g in rng.choice(groups, size=len(groups) // 2, replace=False))
        rows += [dict(group=g, family=family, role='development' if g in development else 'confirmation') for g in groups]
    pd.DataFrame(rows).to_csv(M11_SPLIT, index=False)
m11_split = pd.read_csv(M11_SPLIT)
print(m11_split.groupby(['family', 'role']).size().unstack())

# Shared by 16.2-16.7, defined here so that every notebook (via 16.0) has them without running 16.2
family_of = manifest64.set_index('group')['family']
code_runs = (read_reimpl_results().query("mode == 'code' and status == 'success'")
             .drop_duplicates(['group', 'seed'], keep='last'))
role = m11_split.set_index('group')['role']

def read_m11(kind):
    files = sorted(RESULTS_DIR.glob(f'm11_{kind}_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

SOURCES = ['oe', 'residual', 'error']
COMBOS = [w for w in np.ndindex(3, 3, 3) if sum(w) > 0]          # weights in {0, 1, 2} for (oe, residual, error)
EPS = 1e-12

def m11_sources(model, series):
    """Raw per-point sources on the training part and on the test part."""
    period, window = series['period'], series['window']
    out = {}
    for part in ['train', 'test']:
        c = predict_components(model, series[part], window, period)
        out[part] = dict(oe=c['oe'], residual=c['residual'], combined=c['combined'],
                         error=np.abs(series[part] - c['reconstruction']), x=series[part])
    return out

def smooth(v, period):
    v = moving_average(v, period)
    v[:period] = 0
    return v

def tail_z(values, reference, tail=0.95):
    """Rarity of each value under the reference (training) values: -log of its upper-tail rank (empirical) up to the
    95th percentile u of the reference; above u, an exponential tail (peaks over threshold): z(u) + (v - u) / beta,
    beta = mean excess of the reference over u. Without the tail, every value above the reference maximum would get
    the same score (saturation) and the ranking of strong anomalies would be lost."""
    ref = np.sort(reference)
    n = len(ref)
    at_least = n - np.searchsorted(ref, values, side='left')
    z = -np.log((1 + at_least) / (n + 1))
    u = np.quantile(ref, tail)
    excess = ref[ref > u] - u
    beta = excess.mean() if len(excess) and excess.mean() > 0 else (ref.std() + EPS)
    z_u = -np.log((1 + n - np.searchsorted(ref, u, side='left')) / (n + 1))
    above = values > u
    z[above] = z_u + (values[above] - u) / beta
    return z

def trailing_sd(x, width):
    return pd.Series(x).shift(1).rolling(width, min_periods=2).std().bfill().fillna(0).to_numpy()

def context_tail(test, train, period, bins=4):
    """Reconstruction error ranked against training errors of the same context (local SD over the previous period)."""
    c_train, c_test = trailing_sd(train['x'], period), trailing_sd(test['x'], period)
    edges = np.quantile(c_train, np.linspace(0, 1, bins + 1)[1:-1])
    b_train, b_test = np.searchsorted(edges, c_train), np.searchsorted(edges, c_test)
    z = np.empty(len(c_test))
    for b in range(bins):
        ref = train['error'][b_train == b]
        if len(ref) < 20:
            ref = train['error']
        mask = b_test == b
        z[mask] = tail_z(test['error'][mask], ref)
    return z

def m11_variants(src, period, combos=COMBOS, b3_weights=None):
    """Scores of all variants. combos: B3 weight sets to compute; b3_weights: the locked set (named 'B3')."""
    tr, te = src['train'], src['test']
    out = {}
    def branch(kind):
        f = (lambda v, ref: np.abs(v - ref.mean())) if kind == 'two' else (lambda v, ref: np.maximum(0, v - ref.mean()))
        r, c = smooth(f(te['error'], tr['error']), period), smooth(f(te['combined'], tr['combined']), period)
        rt, ct = smooth(f(tr['error'], tr['error']), period), smooth(f(tr['combined'], tr['combined']), period)
        return r, c, rt.std() + EPS, ct.std() + EPS
    r, c, sr, sc = branch('two')
    out['B0 CoAD'] = (r + c) / 2
    out['B1 calibrated'] = (r / sr + c / sc) / 2
    r1, c1, sr1, sc1 = branch('one')
    out['B1b one-sided'] = (r1 / sr1 + c1 / sc1) / 2
    z = {s: tail_z(te[s], tr[s]) for s in SOURCES}
    out['B2 tail'] = smooth((z['oe'] + z['residual'] + z['error']) / 3, period)
    for w in combos:
        out[f'B3 w={w}'] = smooth(sum(wi * z[s] for wi, s in zip(w, SOURCES)) / sum(w), period)
    if b3_weights is not None:
        out['B3 weighted tail'] = smooth(sum(wi * z[s] for wi, s in zip(b3_weights, SOURCES)) / sum(b3_weights), period)
    out['B4 context tail'] = smooth((z['oe'] + z['residual'] + context_tail(te, tr, period)) / 3, period)
    z_train = {s: tail_z(tr[s], tr[s]) for s in SOURCES}
    f_test, f_train = sum(z.values()) / 3, sum(z_train.values()) / 3
    parts = []
    for length in [1, max(2, period // 4), period]:
        g_test = f_test if length == 1 else moving_average(f_test, length)
        g_train = f_train if length == 1 else moving_average(f_train, length)
        parts.append(tail_z(g_test, g_train))
    b5 = 0.5 * parts[0] + 0.5 * np.mean(parts, axis=0)
    b5[:period] = 0
    out['B5 multi-length'] = b5
    saturation = float((te['error'] > tr['error'].max()).mean())     # share of test errors above every training error
    return out, saturation

def code_model(run):
    model = CoADReimpl(faithful=True).to(DEVICE)
    model.load_state_dict(torch.load(RUNS / run.run_id / 'model.pt', map_location=DEVICE))
    return model

In [ ]:
M11_DEV = RESULTS_DIR / f'm11_dev_{SESSION_TAG}.csv'
M11_DEV_COLUMNS = ['run_id', 'group', 'seed', 'family', 'variant', 'auc_pr', 'saturation']
M11_DEV_SEEDS = SEEDS                    # e.g. [46] in one session and [47, 48] in another

done = read_m11('dev')
done = set(zip(done['group'].astype(int), done['seed'].astype(int))) if len(done) else set()
todo = [r for r in code_runs.itertuples() if role.get(int(r.group)) == 'development' and int(r.seed) in M11_DEV_SEEDS
        and (int(r.group), int(r.seed)) not in done]
print(len(todo), 'development models to do')
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    variants, saturation = m11_variants(m11_sources(code_model(run), series), series['period'])
    gap = float(np.abs(variants['B0 CoAD'] - np.load(RUNS / run.run_id / 'scores.npy')).max())
    assert gap < 1e-4, f'{run.run_id}: B0 differs from scores.npy by {gap:.1e}'
    label = np.asarray(series['label'])
    rows = [dict(run_id=run.run_id, group=group, seed=seed, family=family_of[group], variant=name,
                 auc_pr=average_precision_score(label, score), saturation=saturation) for name, score in variants.items()]
    pd.DataFrame(rows).reindex(columns=M11_DEV_COLUMNS).to_csv(M11_DEV, mode='a', header=not M11_DEV.exists(), index=False)
    print(f'[{i}/{len(todo)}] group {group} seed {seed}: AUC-PR B0 {rows[0]["auc_pr"]:.3f} B1 {rows[1]["auc_pr"]:.3f}', flush=True)

In [ ]:
dev = read_m11('dev').drop_duplicates(['group', 'seed', 'variant'], keep='last')
n_expected = int(sum(role.get(int(r.group)) == 'development' for r in code_runs.itertuples()))
n_done = dev.drop_duplicates(['group', 'seed']).shape[0]
assert n_done == n_expected, f'{n_done} of {n_expected} development models done: finish 16.2 (every session) before locking'
per_series = dev.groupby(['variant', 'group'])['auc_pr'].mean().unstack(0)
summary = (100 * per_series).mean().sort_values(ascending=False)
named = ['B0 CoAD', 'B1 calibrated', 'B1b one-sided', 'B2 tail', 'B4 context tail', 'B5 multi-length']
print(f'{per_series.shape[0]} development series, {dev["seed"].nunique()} seeds')
print('Development AUC-PR (x100), named variants:')
display(summary[named].round(2).to_frame('AUC-PR'))
b3 = summary[[v for v in summary.index if v.startswith('B3 w=')]]
print('Best B3 weight sets (oe, residual, error):')
display(b3.head(5).round(2).to_frame('AUC-PR'))
best = tuple(int(c) for c in b3.index[0].split('=')[1].strip('()').split(','))
locked = {'B3_weights': best, 'variants': ['B1 calibrated', 'B1b one-sided', 'B2 tail', 'B3 weighted tail',
                                          'B4 context tail', 'B5 multi-length'],
          'development_series': sorted(int(g) for g in per_series.index),
          'saturation_mean': float(dev['saturation'].mean())}
(REPORTS / 'm11_locked.json').write_text(json.dumps(locked, indent=2))
print('Locked:', locked['B3_weights'], '| share of test errors above every training error (mean):',
      round(locked['saturation_mean'], 4))

In [ ]:
M11_CONFIRM = RESULTS_DIR / f'm11_confirm_{SESSION_TAG}.csv'
M11_CONFIRM_COLUMNS = ['run_id', 'group', 'seed', 'family', 'variant'] + METRIC_KEYS + ['top_k_normal_share', 'seconds']
M11_CONFIRM_SEEDS = SEEDS                # split between sessions, e.g. [46, 47] in notebook A and [48] in notebook B
assert (REPORTS / 'm11_locked.json').exists(), 'Run 16.3 first (in one session): the variants must be locked before confirmation'
locked = json.loads((REPORTS / 'm11_locked.json').read_text())
done = read_m11('confirm')
done = set(zip(done['group'].astype(int), done['seed'].astype(int))) if len(done) else set()
todo = [r for r in code_runs.itertuples() if role.get(int(r.group)) == 'confirmation' and int(r.seed) in M11_CONFIRM_SEEDS
        and (int(r.group), int(r.seed)) not in done]
print(len(todo), 'confirmation models to do | locked B3 weights', locked['B3_weights'])
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    start = time.time()
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    variants, _ = m11_variants(m11_sources(code_model(run), series), series['period'], combos=[],
                               b3_weights=tuple(locked['B3_weights']))
    gap = float(np.abs(variants['B0 CoAD'] - np.load(RUNS / run.run_id / 'scores.npy')).max())
    assert gap < 1e-4, f'{run.run_id}: B0 differs from scores.npy by {gap:.1e}'
    rows = []
    for name in locked['variants']:
        row = dict(run_id=run.run_id, group=group, seed=seed, family=family_of[group], variant=name,
                   **score_metrics(variants[name], series),
                   top_k_normal_share=alarm_stats(series, variants[name])['top_k_normal_share'])
        rows.append(row)
    for row in rows:
        row['seconds'] = round(time.time() - start, 1)
    pd.DataFrame(rows).reindex(columns=M11_CONFIRM_COLUMNS).to_csv(M11_CONFIRM, mode='a', header=not M11_CONFIRM.exists(), index=False)
    print(f"[{i}/{len(todo)}] group {group} seed {seed}: VUS-PR " + ' | '.join(
        f"{r['variant'].split()[0]} {100 * r['VUS_PR']:.2f}" for r in rows) + f" | {rows[0]['seconds']:.0f} s", flush=True)

In [ ]:
conf = read_m11('confirm').drop_duplicates(['group', 'seed', 'variant'], keep='last')
b0 = code_runs[code_runs['group'].isin(conf['group'].unique())].assign(variant='B0 CoAD')
b0['top_k_normal_share'] = np.nan
allv = pd.concat([b0, conf], ignore_index=True).astype({'group': int, 'seed': int})
order = ['B0 CoAD'] + locked['variants']
TABLE1 = {'F1': 49.13, 'AUC-PR': 43.66, 'R-AUC-PR': 40.50, 'VUS-PR': 39.83}
print(f"{allv['group'].nunique()} confirmation series, seeds {sorted(allv['seed'].unique())}")
rows = []
for v in order:
    d = allv[allv['variant'] == v]
    row = {'variant': v}
    for key, name in METRICS4.items():
        row[name] = 100 * d.groupby('seed')[key].mean().mean()
    row['VUS-PR SD (seeds)'] = (100 * d.groupby('seed')['VUS_PR'].mean()).std()
    row['top-k normal share'] = d['top_k_normal_share'].mean()
    rows.append(row)
table = pd.DataFrame(rows).set_index('variant')
table.loc['Table 1 (64 series, paper)'] = TABLE1
display(table.round(3))

if 'holm' not in globals():                       # defined in 9.3; repeated here so 16.5 runs without it
    def holm(pvalues):
        pvalues = np.asarray(pvalues); adjusted, running = np.empty(len(pvalues)), 0.0
        for i, idx in enumerate(np.argsort(pvalues)):
            running = max(running, (len(pvalues) - i) * pvalues[idx]); adjusted[idx] = min(1.0, running)
        return adjusted
per = 100 * allv.pivot_table(index='group', columns='variant', values='VUS_PR', aggfunc='mean')
tests = []
for reference in ['B0 CoAD', 'B1 calibrated']:
    for v in order:
        if v in ('B0 CoAD', reference):
            continue
        d = per[v] - per[reference]
        tests.append({'vs': reference, 'variant': v, 'difference': d.mean(), 'higher': f'{int((d > 0).sum())}/{len(d)}',
                      'p (one-sided)': wilcoxon(d, alternative='greater').pvalue if (d != 0).any() else np.nan})
tests = pd.DataFrame(tests)
for reference in ['B0 CoAD', 'B1 calibrated']:
    m = tests['vs'] == reference
    tests.loc[m, 'p Holm'] = holm(tests.loc[m, 'p (one-sided)'].fillna(1).to_numpy())
display(tests.round(4))
print('VUS-PR by family (confirmation):')
display(per.groupby(family_of.reindex(per.index).values).mean()[order].round(2))
print('VUS-PR gain over B1 per seed:')
ps = 100 * allv.pivot_table(index=['group', 'seed'], columns='variant', values='VUS_PR')
display(ps[order[2:]].sub(ps['B1 calibrated'], axis=0).groupby(level='seed').mean().round(2))

primary = per['B3 weighted tail'] - per['B1 calibrated']
p = wilcoxon(primary, alternative='greater').pvalue
print(f'Primary prediction (locked B3 > B1 on confirmation, VUS-PR): {primary.mean():+.2f}, higher on '
      f'{int((primary > 0).sum())}/{len(primary)}, one-sided p = {p:.3f} -> ' + ('HOLDS' if p < 0.05 and primary.mean() > 0 else 'FAILS'))
table.to_csv(REPORTS / 'm11_confirmation.csv'); tests.to_csv(REPORTS / 'm11_tests.csv', index=False)

In [ ]:
# Self-contained after 16.1: 16.2 is not needed, so this part can run in a second notebook in parallel
family_of = manifest64.set_index('group')['family']
if 'code_runs' not in globals():
    code_runs = (read_reimpl_results().query("mode == 'code' and status == 'success'")
                 .drop_duplicates(['group', 'seed'], keep='last'))
M11_SPIKE = RESULTS_DIR / f'm11_spike_{SESSION_TAG}.csv'
SPIKE_LENGTHS, SPIKE_OFFSETS, SPIKE_AMPLITUDES = [1, 2, 4, 8, 16], [0, 3, 5, 7], [3, 1]   # x training SD; finished amplitudes are skipped
SPIKE_SIGNALS = ['oe', 'residual', 'error', 'reconstruct', 'classify', 'full_unsmoothed', 'full']
M11_SPIKE_COLUMNS = (['group', 'seed', 'family', 'amplitude', 'length', 'offset', 'position']
                     + [f'{s}_peak' for s in SPIKE_SIGNALS])

def spike_positions(series):
    window, period, n = series['window'], series['period'], len(series['test'])
    blocked = np.zeros(n, bool)
    for a, b in segments(series['label']):
        blocked[max(0, a - window - max(SPIKE_LENGTHS) - PATCH):b + window] = True
    lo, hi = max(2 * window, period) + PATCH, n - 2 * window - max(SPIKE_LENGTHS) - PATCH
    candidates = [s for s in range(lo, hi) if not blocked[s]]
    if not candidates:
        return []
    rng = np.random.default_rng(2026 + series['group'])
    return sorted(int(p) for p in rng.choice(candidates, size=min(4, len(candidates)), replace=False))

def grid_start(series, position, offset):
    """Start of a spike at `offset` within the patch grid of the last test window starting at or before `position`."""
    window, stride, n = series['window'], series['period'], len(series['test'])
    pad = 0 if stride == 1 or (n - window) % stride == 0 else stride - (n - window) % stride
    starts = np.arange(0, pad + n - window + 1, stride) - pad
    w = starts[starts <= position].max()
    return int(position - (position - w) % PATCH + offset)

SPIKE_GROUPS = []
for family, d in manifest64.groupby('family', sort=False):
    chosen = [int(g) for g in d['group'] if spike_positions(load_series(int(g)))][:2]
    print(f'{family}: spike series {chosen}' + ('' if chosen else ' (no normal stretch long enough)'))
    SPIKE_GROUPS += chosen

def read_spike():
    files = sorted(RESULTS_DIR.glob('m11_spike_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

done = read_spike()
done = set(zip(done['group'].astype(int), done['seed'].astype(int), done['amplitude'].astype(float))) if len(done) else set()
todo = [r for r in code_runs.itertuples() if int(r.group) in SPIKE_GROUPS
        and any((int(r.group), int(r.seed), float(a)) not in done for a in SPIKE_AMPLITUDES)]
print(len(todo), 'models for the spike test')
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    window, period = series['window'], series['period']
    model = code_model(run)
    base, train_parts = model_signals(model, series)
    label = np.asarray(series['label'])
    keep = np.ones(len(label), bool)
    for a, b in segments(label):
        keep[max(0, a - window):b + window] = False
    keep[:period] = False
    pcts = {s: percentile_against(base[s][keep]) for s in SPIKE_SIGNALS}
    sd = series['train'].std()
    rows = []
    for position in spike_positions(series):
        for amp in [a for a in SPIKE_AMPLITUDES if (group, seed, float(a)) not in done]:
            for length in SPIKE_LENGTHS:
                for offset in SPIKE_OFFSETS:
                    start = grid_start(series, position, offset)
                    values = series['test'].copy()
                    values[start:start + length] += amp * sd
                    a, b = start - 2 * window, start + length + 2 * window
                    signals, _ = model_signals(model, series, values=values[a:b], train_parts=train_parts)
                    s0 = start - a
                    row = dict(group=group, seed=seed, family=family_of[group], amplitude=amp, length=length,
                               offset=offset, position=position)
                    for s in SPIKE_SIGNALS:
                        row[f'{s}_peak'] = float(pcts[s](np.array([signals[s][s0:s0 + length].max()]))[0])
                    rows.append(row)
    if rows:
        pd.DataFrame(rows).reindex(columns=M11_SPIKE_COLUMNS).to_csv(M11_SPIKE, mode='a', header=not M11_SPIKE.exists(), index=False)
    print(f'[{i}/{len(todo)}] group {group} seed {seed}: {len(rows)} spikes', flush=True)

In [ ]:
sp = read_spike().drop_duplicates(['group', 'seed', 'amplitude', 'length', 'offset', 'position'], keep='last')
found = sp.assign(**{s: sp[f'{s}_peak'] >= 0.99 for s in SPIKE_SIGNALS})
print(f"{sp['group'].nunique()} series, {len(sp)} spikes, amplitudes {sorted(sp['amplitude'].unique())}")
for amp, f in found.groupby('amplitude'):
    print()
    print(f'===== amplitude {amp} x training SD')
    by_len = f.groupby('length')[SPIKE_SIGNALS].mean()
    print('Share of spikes found, by length:')
    display(by_len.round(3))
    print('Share found by OE, by length and offset:')
    display(f.pivot_table(index='length', columns='offset', values='oe').round(3))
    print('By family:')
    display(f.groupby(['family', 'length'])[['oe', 'error', 'full_unsmoothed', 'full']].mean().round(3))
    s1 = bool(by_len['oe'].is_monotonic_increasing and by_len['oe'].iloc[-1] - by_len['oe'].iloc[0] >= 0.2)
    short = f[f['length'] <= 4]
    s2 = short['full_unsmoothed'].mean() - short['full'].mean()
    eight = f[f['length'] == 8]
    s3 = eight.loc[eight['offset'] == 0, 'oe'].mean() - eight.loc[eight['offset'] == 5, 'oe'].mean()
    print(f"S1 (OE found-rate rises with length by >= 0.20): {by_len['oe'].round(3).tolist()} -> " + ('HOLDS' if s1 else 'FAILS'))
    print(f'S2 (smoothing loss on 1-4-point spikes >= 0.20): {s2:+.3f} -> ' + ('HOLDS' if s2 >= 0.2 else 'FAILS'))
    print(f'S3 (8-point spike: OE offset 0 - offset 5 > 0): {s3:+.3f} -> ' + ('HOLDS' if s3 > 0 else 'FAILS'))
found.groupby(['amplitude', 'length'])[SPIKE_SIGNALS].mean().to_csv(REPORTS / 'm11_spike_by_length.csv')

In [ ]:
# Self-contained after 16.1 (16.6 is not needed)
family_of = manifest64.set_index('group')['family']
if 'code_runs' not in globals():
    code_runs = (read_reimpl_results().query("mode == 'code' and status == 'success'")
                 .drop_duplicates(['group', 'seed'], keep='last'))
M11_DENSITY = RESULTS_DIR / f'm11_density_{SESSION_TAG}.csv'
DENSITY_AMPLITUDES, DENSITY_GAPS = [0.5, 1], [4, 0.4, 0.2, 0.1]     # x training SD; gap between spikes in periods
DENSITY_SIGNALS = ['error', 'oe', 'full_unsmoothed', 'full', 'calibrated']
M11_DENSITY_COLUMNS = (['group', 'seed', 'family', 'amplitude', 'gap_periods', 'gap', 'n_spikes']
                       + [f'{s}_{k}' for s in DENSITY_SIGNALS for k in ['found_own', 'found_clean', 'normal_above']])

def normal_mask(series):
    """Normal test points used as the percentile reference: away from labelled anomalies, after the first period."""
    window, period = series['window'], series['period']
    keep = np.ones(len(series['test']), bool)
    for a, b in segments(series['label']):
        keep[max(0, a - window):b + window] = False
    keep[:period] = False
    return keep

def gap_points(series, gap_periods):
    return max(5, int(round(gap_periods * series['period'])))

def density_positions(series, gap_periods):
    gap = gap_points(series, gap_periods)
    window, period, n = series['window'], series['period'], len(series['test'])
    allowed = normal_mask(series)
    phase = int(np.random.default_rng(2026 + series['group']).integers(gap))
    return np.array([p for p in range(period + window + phase, n - window, gap) if allowed[p]], int)

DENSITY_GROUPS = []
for family, d in manifest64.groupby('family', sort=False):
    chosen = [int(g) for g in d['group'] if len(density_positions(load_series(int(g)), max(DENSITY_GAPS))) >= 3][:2]
    print(f'{family}: density series {chosen}')
    DENSITY_GROUPS += chosen

def read_density():
    files = sorted(RESULTS_DIR.glob('m11_density_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

done = read_density()
done = (set(zip(done['group'].astype(int), done['seed'].astype(int), done['amplitude'].astype(float),
                done['gap_periods'].astype(float))) if len(done) else set())
keys = lambda g, s: [(g, s, float(a), float(q)) for a in DENSITY_AMPLITUDES for q in DENSITY_GAPS]
todo = [r for r in code_runs.itertuples() if int(r.group) in DENSITY_GROUPS
        and any(k not in done for k in keys(int(r.group), int(r.seed)))]
print(len(todo), 'models for the density test')
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    model = code_model(run)
    # a copy, so the first period is processed as in the injected runs (same train SD for `calibrated`)
    base, train_parts = model_signals(model, series, values=series['test'].copy())
    keep = normal_mask(series)
    clean = {s: percentile_against(base[s][keep]) for s in DENSITY_SIGNALS}
    clean99 = {s: np.quantile(base[s][keep], 0.99) for s in DENSITY_SIGNALS}
    sd = series['train'].std()
    rows = []
    for _, _, amp, gap in [k for k in keys(group, seed) if k not in done]:
        pos = density_positions(series, gap)
        values = series['test'].copy()
        values[pos] += amp * sd
        signals, _ = model_signals(model, series, values=values, train_parts=train_parts)
        own_mask = keep.copy()
        own_mask[pos] = False
        row = dict(group=group, seed=seed, family=family_of[group], amplitude=amp, gap_periods=gap,
                   gap=gap_points(series, gap), n_spikes=len(pos))
        for s in DENSITY_SIGNALS:
            sig = signals[s]
            own = percentile_against(sig[own_mask])
            row[f'{s}_found_own'] = float((own(sig[pos]) >= 0.99).mean())
            row[f'{s}_found_clean'] = float((clean[s](sig[pos]) >= 0.99).mean())
            row[f'{s}_normal_above'] = float((sig[own_mask] > clean99[s]).mean())
        rows.append(row)
    if rows:
        (pd.DataFrame(rows).reindex(columns=M11_DENSITY_COLUMNS)
         .to_csv(M11_DENSITY, mode='a', header=not M11_DENSITY.exists(), index=False))
    print(f'[{i}/{len(todo)}] group {group} seed {seed}: {len(rows)} conditions', flush=True)

In [ ]:
dn = read_density().drop_duplicates(['group', 'seed', 'amplitude', 'gap_periods'], keep='last')
print(f"{dn['group'].nunique()} series, {len(dn)} conditions; spikes per series (median by gap in periods):",
      dn.groupby('gap_periods')['n_spikes'].median().to_dict())
for kind in ['found_own', 'found_clean', 'normal_above']:
    print()
    print(kind, '(mean over series and seeds); rows: amplitude, gap in periods')
    t = dn.groupby(['amplitude', 'gap_periods'])[[f'{s}_{kind}' for s in DENSITY_SIGNALS]].mean()
    t.columns = DENSITY_SIGNALS
    display(t.sort_index(ascending=[True, False]).round(3))
print('Final score found_own by family:')
display(dn.pivot_table(index=['amplitude', 'gap_periods'], columns='family', values='full_found_own')
        .sort_index(ascending=[True, False]).round(3))
m = dn.groupby(['amplitude', 'gap_periods']).mean(numeric_only=True)
d_own = m.loc[(1, 4), 'full_found_own'] - m.loc[(1, 0.2), 'full_found_own']
d_clean = m.loc[(1, 4), 'full_found_clean'] - m.loc[(1, 0.2), 'full_found_clean']
d3 = m.loc[(0.5, 0.2), 'full_found_own']
print()
print(f'D1 (1 SD, final found_own 4 periods - 0.2 period >= 0.20): {d_own:+.3f} -> ' + ('HOLDS' if d_own >= 0.2 else 'FAILS'))
print(f'D2 (D1 holds and drop of found_clean < half the drop of found_own): {d_clean:+.3f} vs {d_own:+.3f} -> '
      + ('HOLDS' if d_own >= 0.2 and d_clean < d_own / 2 else 'FAILS'))
print(f'D3 (0.5 SD, 0.2 period, final found_own <= 0.10; real 1-point events 0.013): {d3:.3f} -> '
      + ('HOLDS' if d3 <= 0.1 else 'FAILS'))
m.to_csv(REPORTS / 'm11_density.csv')

In [ ]:
from scipy.ndimage import maximum_filter1d

def m12_operators(period):
    widths = {'P/8': max(2, period // 8), 'P/4': max(2, period // 4), 'P/2': max(2, period // 2), 'P': period}
    ops = {'none': lambda v: v.copy()}
    for name in ['P/4', 'P/2', 'P']:
        ops[f'mean {name}'] = (lambda w: lambda v: moving_average(v, w))(widths[name])
    for name in ['P/8', 'P/4', 'P/2', 'P']:
        ops[f'max {name}'] = (lambda w: lambda v: maximum_filter1d(v, size=w, mode='nearest'))(widths[name])
    return ops
M12_OPS = list(m12_operators(100)) + ['mix']
M12_A_PRIORI = ('max P/4', 'mean P')                # (error, classify), fixed from 16.11 before running

def m12_branches(src, period):
    """Calibrated branch scores for every operator: {op: (test score / train SD)} for the error and classify branches."""
    tr, te = src['train'], src['test']
    out = {}
    for branch, key in [('error', 'error'), ('classify', 'combined')]:
        mu = tr[key].mean()
        dev_test, dev_train = np.abs(te[key] - mu), np.abs(tr[key] - mu)
        scaled = {}
        for name, op in m12_operators(period).items():
            a, b = op(dev_test), op(dev_train)
            a[:period] = 0
            b[:period] = 0
            scaled[name] = a / (b.std() + EPS)
        scaled['mix'] = (scaled['mean P'] + scaled['none']) / 2
        out[branch] = scaled
    return out

def m12_name(pair):
    return f'error={pair[0]} | classify={pair[1]}'

def m12_scores(branches, pairs):
    return {m12_name(p): (branches['error'][p[0]] + branches['classify'][p[1]]) / 2 for p in pairs}

def read_m12(kind):
    files = sorted(RESULTS_DIR.glob(f'm12_{kind}_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

M12_PAIRS = [(a, b) for a in M12_OPS for b in M12_OPS]
# Two notebooks at the same time: 'gpu-a' takes the models with (group + seed) even, 'gpu-b' the odd ones (a fixed
# split, so the two never take the same model); any other SESSION_TAG runs every model
M12_PARTS = {'gpu-a': 0, 'gpu-b': 1}
def m12_mine(group, seed):
    part = M12_PARTS.get(SESSION_TAG)
    return part is None or (int(group) + int(seed)) % 2 == part
print(len(M12_OPS), 'operators per branch,', len(M12_PAIRS), 'pairs | a priori:', m12_name(M12_A_PRIORI),
      '| SESSION_TAG', SESSION_TAG, '->', 'half', M12_PARTS[SESSION_TAG] if SESSION_TAG in M12_PARTS else 'all models')

In [ ]:
M12_DEV = RESULTS_DIR / f'm12_dev_{SESSION_TAG}.csv'
M12_DEV_COLUMNS = ['run_id', 'group', 'seed', 'family', 'variant', 'auc_pr']
M12_DEV_SEEDS = SEEDS                    # the split between notebooks is by SESSION_TAG (m12_mine, 16.12)

done = read_m12('dev')
done = set(zip(done['group'].astype(int), done['seed'].astype(int))) if len(done) else set()
todo = [r for r in code_runs.itertuples() if role.get(int(r.group)) == 'development' and int(r.seed) in M12_DEV_SEEDS
        and m12_mine(r.group, r.seed) and (int(r.group), int(r.seed)) not in done]
print(len(todo), 'development models to do')
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    src = m11_sources(code_model(run), series)
    scores = m12_scores(m12_branches(src, series['period']), M12_PAIRS)
    b1 = m11_variants(src, series['period'], combos=[])[0]['B1 calibrated']
    gap = float(np.abs(scores[m12_name(('mean P', 'mean P'))] - b1).max())
    assert gap < 1e-6 * (1 + np.abs(b1).max()), f'{run.run_id}: mean P | mean P differs from B1 by {gap:.1e}'
    label = np.asarray(series['label'])
    rows = [dict(run_id=run.run_id, group=group, seed=seed, family=family_of[group], variant=name,
                 auc_pr=average_precision_score(label, score)) for name, score in scores.items()]
    pd.DataFrame(rows).reindex(columns=M12_DEV_COLUMNS).to_csv(M12_DEV, mode='a', header=not M12_DEV.exists(), index=False)
    by = {r['variant']: r['auc_pr'] for r in rows}
    print(f"[{i}/{len(todo)}] group {group} seed {seed}: AUC-PR B1 {by[m12_name(('mean P', 'mean P'))]:.3f} "
          f"a priori {by[m12_name(M12_A_PRIORI)]:.3f} best {max(by.values()):.3f}", flush=True)

In [ ]:
dev12 = read_m12('dev').drop_duplicates(['group', 'seed', 'variant'], keep='last')
n_expected = int(sum(role.get(int(r.group)) == 'development' for r in code_runs.itertuples()))
n_done = dev12.drop_duplicates(['group', 'seed']).shape[0]
assert n_done == n_expected, f'{n_done} of {n_expected} development models done: finish 16.13 (every session) before locking'
per12 = dev12.groupby(['variant', 'group'])['auc_pr'].mean().unstack(0)
summary12 = (100 * per12).mean().sort_values(ascending=False)
B1_NAME, AP_NAME = m12_name(('mean P', 'mean P')), m12_name(M12_A_PRIORI)
print(f'{per12.shape[0]} development series, {dev12["seed"].nunique()} seeds')
print('Top 10 pairs (development AUC-PR x 100):')
display(summary12.head(10).round(2).to_frame('AUC-PR'))
print(f'B1 (mean P | mean P): {summary12[B1_NAME]:.2f} | a priori: {summary12[AP_NAME]:.2f}')
print('Development AUC-PR by error operator (classify = mean P) and by classify operator (error = mean P):')
display(pd.DataFrame({'error op (classify=mean P)': [summary12[m12_name((o, 'mean P'))] for o in M12_OPS],
                      'classify op (error=mean P)': [summary12[m12_name(('mean P', o))] for o in M12_OPS]},
                     index=M12_OPS).round(2))
fam12 = (100 * per12[[summary12.index[0], B1_NAME, AP_NAME]]).groupby(family_of.reindex(per12.index).values).mean()
print('Development AUC-PR by family:')
display(fam12.round(2))
best12 = summary12.index[0]
pair = tuple(part.split('=')[1] for part in best12.split(' | '))
locked12 = {'locked_pair': pair, 'a_priori_pair': M12_A_PRIORI, 'development_series': sorted(int(g) for g in per12.index)}
(REPORTS / 'm12_locked.json').write_text(json.dumps(locked12, indent=2))
print('Locked:', best12)

In [ ]:
M12_CONFIRM = RESULTS_DIR / f'm12_confirm_{SESSION_TAG}.csv'
M12_CONFIRM_COLUMNS = ['run_id', 'group', 'seed', 'family', 'variant'] + METRIC_KEYS + ['top_k_normal_share', 'seconds']
M12_CONFIRM_SEEDS = SEEDS                # the split between notebooks is by SESSION_TAG (m12_mine, 16.12)
assert (REPORTS / 'm12_locked.json').exists(), 'Run 16.14 first: the pair must be locked before confirmation'
locked12 = json.loads((REPORTS / 'm12_locked.json').read_text())
m12_conf_variants = {'M12 locked': tuple(locked12['locked_pair']), 'M12 a priori': tuple(locked12['a_priori_pair'])}
done = read_m12('confirm')
done = set(zip(done['group'].astype(int), done['seed'].astype(int))) if len(done) else set()
todo = [r for r in code_runs.itertuples() if role.get(int(r.group)) == 'confirmation' and int(r.seed) in M12_CONFIRM_SEEDS
        and m12_mine(r.group, r.seed) and (int(r.group), int(r.seed)) not in done]
print(len(todo), 'confirmation models to do |', {k: m12_name(v) for k, v in m12_conf_variants.items()})
session_start = time.time()
for i, run in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    start = time.time()
    group, seed = int(run.group), int(run.seed)
    series = load_series(group)
    branches = m12_branches(m11_sources(code_model(run), series), series['period'])
    rows, cache = [], {}
    for name, p in m12_conf_variants.items():
        score = m12_scores(branches, [p])[m12_name(p)]
        if p not in cache:                                  # same pair locked and a priori: evaluate once
            cache[p] = dict(**score_metrics(score, series),
                            top_k_normal_share=alarm_stats(series, score)['top_k_normal_share'])
        rows.append(dict(run_id=run.run_id, group=group, seed=seed, family=family_of[group], variant=name, **cache[p]))
    for row in rows:
        row['seconds'] = round(time.time() - start, 1)
    pd.DataFrame(rows).reindex(columns=M12_CONFIRM_COLUMNS).to_csv(M12_CONFIRM, mode='a', header=not M12_CONFIRM.exists(), index=False)
    print(f"[{i}/{len(todo)}] group {group} seed {seed}: VUS-PR " + ' | '.join(
        f"{r['variant']} {100 * r['VUS_PR']:.2f}" for r in rows) + f" | {rows[0]['seconds']:.0f} s", flush=True)

In [ ]:
conf12 = read_m12('confirm').drop_duplicates(['group', 'seed', 'variant'], keep='last')
locked12 = json.loads((REPORTS / 'm12_locked.json').read_text())
b1 = read_m11('confirm').drop_duplicates(['group', 'seed', 'variant'], keep='last').query("variant == 'B1 calibrated'")
b0 = code_runs[code_runs['group'].isin(conf12['group'].unique())].assign(variant='B0 CoAD')
b0['top_k_normal_share'] = np.nan
all12 = pd.concat([b0, b1, conf12], ignore_index=True).astype({'group': int, 'seed': int})
all12 = all12[all12['group'].isin(conf12['group'].unique()) & all12['seed'].isin(conf12['seed'].unique())]
order12 = ['B0 CoAD', 'B1 calibrated', 'M12 locked', 'M12 a priori']
print(f"{conf12['group'].nunique()} confirmation series, seeds {sorted(conf12['seed'].unique())} | locked "
      f"{m12_name(locked12['locked_pair'])} | a priori {m12_name(locked12['a_priori_pair'])}")
n_models = all12.groupby('variant')[['group', 'seed']].apply(lambda d: len(d.drop_duplicates()))
assert n_models.nunique() == 1, f'unequal model counts: {n_models.to_dict()} (finish 16.15 in every session)'
rows = []
for v in order12:
    d = all12[all12['variant'] == v]
    row = {'variant': v}
    for key, name in METRICS4.items():
        row[name] = 100 * d.groupby('seed')[key].mean().mean()
    row['VUS-PR SD (seeds)'] = (100 * d.groupby('seed')['VUS_PR'].mean()).std()
    row['top-k normal share'] = d['top_k_normal_share'].mean()
    rows.append(row)
table12 = pd.DataFrame(rows).set_index('variant')
table12.loc['Table 1 (64 series, paper)'] = {'F1': 49.13, 'AUC-PR': 43.66, 'R-AUC-PR': 40.50, 'VUS-PR': 39.83}
display(table12.round(3))
if 'holm' not in globals():                       # defined in 9.3; repeated so 16.16 runs without it
    def holm(pvalues):
        pvalues = np.asarray(pvalues); adjusted, running = np.empty(len(pvalues)), 0.0
        for i, idx in enumerate(np.argsort(pvalues)):
            running = max(running, (len(pvalues) - i) * pvalues[idx]); adjusted[idx] = min(1.0, running)
        return adjusted
per12c = 100 * all12.pivot_table(index='group', columns='variant', values='VUS_PR', aggfunc='mean')
tests12 = []
for reference in ['B0 CoAD', 'B1 calibrated']:
    for v in order12:
        if v in ('B0 CoAD', reference):
            continue
        d = per12c[v] - per12c[reference]
        tests12.append({'vs': reference, 'variant': v, 'difference': d.mean(), 'higher': f'{int((d > 0).sum())}/{len(d)}',
                        'p (one-sided)': wilcoxon(d, alternative='greater').pvalue if (d != 0).any() else np.nan})
tests12 = pd.DataFrame(tests12)
for reference in ['B0 CoAD', 'B1 calibrated']:
    m = tests12['vs'] == reference
    tests12.loc[m, 'p Holm'] = holm(tests12.loc[m, 'p (one-sided)'].fillna(1).to_numpy())
display(tests12.round(4))
print('VUS-PR by family (confirmation):')
display(per12c.groupby(family_of.reindex(per12c.index).values).mean()[order12].round(2))
print('VUS-PR gain over B1 per seed:')
ps12 = 100 * all12.pivot_table(index=['group', 'seed'], columns='variant', values='VUS_PR')
display(ps12[order12[2:]].sub(ps12['B1 calibrated'], axis=0).groupby(level='seed').mean().round(2))
for v, label in [('M12 locked', 'Primary (locked pair > B1)'), ('M12 a priori', 'Secondary (a-priori pair > B1)')]:
    d = per12c[v] - per12c['B1 calibrated']
    p = wilcoxon(d, alternative='greater').pvalue if (d != 0).any() else 1.0
    print(f'{label}, VUS-PR: {d.mean():+.2f}, higher on {int((d > 0).sum())}/{len(d)}, one-sided p = {p:.3f} -> '
          + ('HOLDS' if p < 0.05 and d.mean() > 0 else 'FAILS'))
table12.to_csv(REPORTS / 'm12_confirmation.csv'); tests12.to_csv(REPORTS / 'm12_tests.csv', index=False)

In [ ]:
import hashlib
from scipy.stats import spearmanr

M13_FILE = STORE / 'eda' / 'm13_external.csv'
M13_EXCLUDED = {'MGAB', 'SED', 'SVDB', 'IOPS', 'TODS'}        # the 64 series of the paper
M13_PER_FAMILY, M13_MIN_FAMILY, M13_MAX_LENGTH = 5, 9, 100_000

def m13_valid(series):
    """Window settings usable by the model (no label values used)."""
    return series['window'] >= 2 * PATCH and len(series['train']) >= series['window'] and series['period'] >= 2

if not M13_FILE.exists():
    files = sorted(RAW.glob('*.csv'))
    catalog = pd.DataFrame([dict(group=int(f.name.split('_')[0]), family=f.name.split('_')[1], file=f.name) for f in files])
    catalog = catalog[~catalog['family'].isin(M13_EXCLUDED)]
    sizes = catalog['family'].value_counts()
    catalog = catalog[catalog['family'].isin(sizes[sizes >= M13_MIN_FAMILY].index)]
    catalog['length'] = [sum(1 for _ in open(RAW / f)) - 1 for f in catalog['file']]
    # TSB-AD-U holds byte-identical copies (Exathlon, YAHOO): keep the lowest id of each content
    catalog['md5'] = [hashlib.md5((RAW / f).read_bytes()).hexdigest() for f in catalog['file']]
    n_before = len(catalog)
    catalog = catalog.sort_values('group').drop_duplicates('md5', keep='first')
    print(n_before - len(catalog), 'duplicate files dropped')
    sizes = catalog['family'].value_counts()
    catalog = catalog[catalog['family'].isin(sizes[sizes >= M13_MIN_FAMILY].index)]
    catalog = catalog[catalog['length'] <= M13_MAX_LENGTH]
    rng = np.random.default_rng(2027)
    rows = []
    for family in sorted(catalog['family'].unique()):
        groups = sorted(catalog.loc[catalog['family'] == family, 'group'])
        kept = []
        for g in rng.permutation(groups):
            try:
                ok = m13_valid(load_series(int(g)))
            except Exception:
                ok = False
            if ok:
                kept.append(int(g))
            if len(kept) == M13_PER_FAMILY:
                break
        rows += [dict(group=g, family=family, length=int(catalog.set_index('group').loc[g, 'length'])) for g in kept]
    pd.DataFrame(rows).to_csv(M13_FILE, index=False)
m13_series = pd.read_csv(M13_FILE)
M13_GROUPS = m13_series['group'].astype(int).tolist()
m13_family = m13_series.set_index('group')['family']
print(len(M13_GROUPS), 'external series:', m13_series.groupby('family').size().to_dict())

M13_RESULTS = RESULTS_DIR / f'm13_external_{SESSION_TAG}.csv'
M13_RUNS = STORE / 'runs' / 'm13'
M13_RUNS.mkdir(parents=True, exist_ok=True)
M13_COLUMNS = (['run_id', 'group', 'seed', 'family', 'variant', 'status', 'error'] + METRIC_KEYS
               + ['top_k_normal_share', 'log_sd_ratio', 'train_seconds'])
M13_SPLIT = {'gpu-a': 0, 'gpu-b': 1}       # two notebooks; any other SESSION_TAG runs every (series, seed)

def m13_mine(group, seed):
    part = M13_SPLIT.get(SESSION_TAG)
    return part is None or (int(group) + int(seed)) % 2 == part

def read_m13():
    files = sorted(RESULTS_DIR.glob('m13_external_*.csv'))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame(columns=M13_COLUMNS)

M13_SEEDS = SEEDS        # [46, 47, 48]; seed 46 is always trained first, so stopping early still leaves complete seeds
mine = [(g, s) for s in M13_SEEDS for g in M13_GROUPS if m13_mine(g, s)]
print('SESSION_TAG', SESSION_TAG, '->', len(mine), 'of', len(M13_GROUPS) * len(SEEDS), '(series, seed) pairs in this notebook')

In [ ]:
assert (REPORTS / 'm12_locked.json').exists(), 'M13 uses the M12 locked pair: m12_locked.json is missing (16.14)'
m13_pair = tuple(json.loads((REPORTS / 'm12_locked.json').read_text())['locked_pair'])
missing = [n for n in ['train_model', 'load_series', 'anomaly_score', 'score_metrics', 'alarm_stats', 'm11_sources',
                       'm11_variants', 'm12_branches', 'm12_scores'] if n not in globals()]
assert not missing, f'not defined: {missing}. Run 16.0 (current version) and 16.12 first'
done = read_m13()
if len(done):
    ok_pairs = set(zip(done.loc[done['status'] == 'success', 'group'].astype(int), done.loc[done['status'] == 'success', 'seed'].astype(int)))
    n_failed = done[done['status'] == 'failed'].groupby(['group', 'seed']).size()
    given_up = {(int(g), int(s)) for (g, s), n in n_failed.items() if n >= 2}     # failed twice: not retried again
    done = ok_pairs | given_up
else:
    done = set()
todo = [(g, s) for g, s in mine if (g, s) not in done]
print(len(todo), 'models to do in this notebook | M12 locked pair:', m12_name(m13_pair))
session_start = time.time()
for i, (group, seed) in enumerate(todo, 1):
    if (time.time() - session_start) / 60 > TIME_BUDGET_MIN:
        print('Time budget reached: re-run this cell to continue')
        break
    run_id = f"{datetime.now(timezone.utc):%Y%m%dT%H%M%S}_m13_{group}_s{seed}"
    base = dict(run_id=run_id, group=group, seed=seed, family=m13_family[group])
    start = time.time()
    try:
        series = load_series(group)
        model, history = train_model(series, faithful=True, seed=seed, log_every=0)
        train_seconds = round(time.time() - start, 1)
        out = M13_RUNS / run_id
        out.mkdir(parents=True)
        torch.save(model.state_dict(), out / 'model.pt')
        history.to_csv(out / 'loss.csv', index=False)
        period = series['period']
        src = m11_sources(model, series)
        b = m11_variants(src, period, combos=[])[0]
        gap = float(np.abs(b['B0 CoAD'] - anomaly_score(model, series)).max())
        assert gap < 1e-4, f'B0 differs from anomaly_score by {gap:.1e}'
        tr = src['train']
        sd_error = smooth(np.abs(tr['error'] - tr['error'].mean()), period).std()
        sd_classify = smooth(np.abs(tr['combined'] - tr['combined'].mean()), period).std()
        log_ratio = float(np.log((sd_error + EPS) / (sd_classify + EPS)))
        scores = {'B0 CoAD': b['B0 CoAD'], 'B1 calibrated': b['B1 calibrated'],
                  'M12 locked': m12_scores(m12_branches(src, period), [m13_pair])[m12_name(m13_pair)]}
        rows = []
        for name, score in scores.items():
            rows.append(dict(base, variant=name, status='success', **score_metrics(score, series),
                             top_k_normal_share=alarm_stats(series, score)['top_k_normal_share'],
                             log_sd_ratio=log_ratio, train_seconds=train_seconds))
        message = ' | '.join(f"{r['variant'].split()[0]} {100 * r['VUS_PR']:.2f}" for r in rows)
    except Exception as error:
        if isinstance(error, NameError):      # a setup problem, not a property of the series: stop, write nothing
            raise
        rows = [dict(base, variant='-', status='failed', error=f'{type(error).__name__}: {error}'[:300])]
        message = 'FAILED: ' + rows[0]['error']
    pd.DataFrame(rows).reindex(columns=M13_COLUMNS).to_csv(M13_RESULTS, mode='a', header=not M13_RESULTS.exists(), index=False)
    print(f'[{i}/{len(todo)}] {m13_family[group]} {group} seed {seed}: VUS-PR {message} | {time.time() - start:.0f} s', flush=True)

In [ ]:
m13 = read_m13()
m13 = pd.concat([m13[m13['status'] == 'failed'],
                 m13[m13['status'] == 'success'].drop_duplicates(['group', 'seed', 'variant'], keep='last')])
m13 = m13[m13['seed'].astype(int).isin(M13_SEEDS)]
ok = m13[m13['status'] == 'success'].astype({'group': int, 'seed': int})
ok_pairs = set(zip(ok['group'], ok['seed']))
bad = m13[(m13['status'] == 'failed') & [(int(g), int(s)) not in ok_pairs for g, s in zip(m13['group'], m13['seed'])]]
n_bad = bad.groupby(['group', 'seed']).size()
failed = bad[[n_bad[(g, s)] >= 2 for g, s in zip(bad['group'], bad['seed'])]].drop_duplicates(['group', 'seed'], keep='last')
n_pairs = len(M13_GROUPS) * len(M13_SEEDS)
n_done = len(ok_pairs) + len(failed)
print(f'Seeds analysed: {M13_SEEDS} | {n_done} of {n_pairs} (series, seed) pairs done, {len(failed)} failed')
if n_done < n_pairs:
    print('WARNING: not finished (run 16.19 in both notebooks); the numbers below are provisional')
if len(failed):
    display(failed[['group', 'seed', 'family', 'error']])
VARIANTS13 = ['B0 CoAD', 'B1 calibrated', 'M12 locked']
complete = ok.groupby('group')['variant'].nunique()
series_ok = complete[complete == len(VARIANTS13)].index
assert len(series_ok) >= 6, f'only {len(series_ok)} series with results for seeds {M13_SEEDS}: run 16.19 first'
rows = []
for v in VARIANTS13:
    d = ok[(ok['variant'] == v) & ok['group'].isin(series_ok)]
    row = {'variant': v}
    for key, name in METRICS4.items():
        row[name] = 100 * d.groupby('seed')[key].mean().mean()
    row['VUS-PR SD (seeds)'] = (100 * d.groupby('seed')['VUS_PR'].mean()).std()
    row['top-k normal share'] = d['top_k_normal_share'].mean()
    rows.append(row)
print(f'{len(series_ok)} series with every variant, seeds {sorted(ok["seed"].unique())}')
display(pd.DataFrame(rows).set_index('variant').round(3))

per13 = 100 * ok[ok['group'].isin(series_ok)].pivot_table(index='group', columns='variant', values='VUS_PR', aggfunc='mean')
def one_sided(diff):
    return wilcoxon(diff, alternative='greater').pvalue if (diff != 0).any() else 1.0
tests13 = []
for v, ref in [('B1 calibrated', 'B0 CoAD'), ('M12 locked', 'B0 CoAD'), ('M12 locked', 'B1 calibrated')]:
    d = per13[v] - per13[ref]
    tests13.append({'variant': v, 'vs': ref, 'difference': d.mean(), 'higher': f'{int((d > 0).sum())}/{len(d)}',
                    'p (one-sided)': one_sided(d)})
display(pd.DataFrame(tests13).round(4))
gain = per13['B1 calibrated'] - per13['B0 CoAD']
by_family = pd.DataFrame({'B0 CoAD': per13['B0 CoAD'], 'B1 - B0': gain, 'M12 - B0': per13['M12 locked'] - per13['B0 CoAD'],
                          'M12 - B1': per13['M12 locked'] - per13['B1 calibrated']}).groupby(m13_family.reindex(per13.index).values).mean()
print('VUS-PR by family (mean over series and seeds):')
display(by_family.round(2))

ratio = ok[ok['variant'] == 'B0 CoAD'].groupby('group')['log_sd_ratio'].mean().abs().reindex(gain.index)
rho, p_rho = spearmanr(ratio, gain)
p_rho = p_rho / 2 if rho > 0 else 1 - p_rho / 2
e1 = one_sided(gain)
e2 = int((by_family['B1 - B0'] > 0).sum())
d4 = per13['M12 locked'] - per13['B0 CoAD']
print()
print(f'E1 (calibrated > CoAD): {gain.mean():+.2f}, higher on {int((gain > 0).sum())}/{len(gain)}, p = {e1:.4f} -> '
      + ('HOLDS' if e1 < 0.05 and gain.mean() > 0 else 'FAILS'))
print(f'E2 (gain positive in >= 7 of {len(by_family)} families): {e2} -> ' + ('HOLDS' if e2 >= 7 else 'FAILS'))
print(f'E3 (Spearman |log SD ratio| vs gain > 0): rho = {rho:+.3f}, one-sided p = {p_rho:.4f} -> '
      + ('HOLDS' if rho > 0 and p_rho < 0.05 else 'FAILS'))
print(f'E4 (M12 locked > CoAD): {d4.mean():+.2f}, higher on {int((d4 > 0).sum())}/{len(d4)}, p = {one_sided(d4):.4f} -> '
      + ('HOLDS' if one_sided(d4) < 0.05 and d4.mean() > 0 else 'FAILS'))
pd.DataFrame(rows).to_csv(REPORTS / 'm13_external.csv', index=False)
by_family.to_csv(REPORTS / 'm13_by_family.csv')

In [ ]:
# Files to put back into the Drive folder CoAD_project (same sub-folders): download to_drive.zip from the Output tab.
# Only this notebook's own result files (tag kaggle-m11), so nothing written on Colab meanwhile is overwritten.
OUT = Path('/kaggle/working/to_drive')
shutil.rmtree(OUT, ignore_errors=True)
(OUT / 'results').mkdir(parents=True)
for csv in (STORE / 'results').glob(f'*{KAGGLE_TAG}*.csv'):
    shutil.copy2(csv, OUT / 'results' / csv.name)
for sub in ['reports'] + ['eda']:
    if (STORE / sub).exists():
        shutil.copytree(STORE / sub, OUT / sub)
shutil.make_archive(str(OUT), 'zip', OUT)
print('to_drive.zip:', sorted(str(p.relative_to(OUT)) for p in OUT.rglob('*.csv')))